# Native catalog, search and MCP

These cells read capabilities and source-owned identities, then reuse the exact saved result through MCP. They start no research agent or execution. The local kernel supplies only Epoch login and service URLs.

In [ ]:
from acceptance_helpers import connect, check_services

epoch = connect()
await check_services()

## 1. Read the shared dataset descriptions

Known constructors need no company lookup. These are the same Backend descriptions available to MCP and the supervisor, including full option meanings.

In [ ]:
catalog = await epoch.dataset_catalog(source_ids=["cash_flow", "mb_vintage_data"], include_saved=False)
print(catalog.overview)
for source in catalog.sources:
    print(source.documentation)

## 2. Look up an instrument by name

The service supplies the actual identifier. The notebook does not make up a ticker.

In [ ]:
from stratifyx.types import DatasetSearchQuery
from acceptance_helpers import identities

found = await epoch.search_dataset([DatasetSearchQuery.model_validate({
    "request_id": "company",
    "subjects": [{"kind": "asset_name", "name": "Microsoft", "asset_type": "Stocks"}],
})])
identities(found)

## 3. Resolve a complete current universe

This is separate from ranked name search. The saved reference represents the whole usable cohort; an acquisition gap must remain visible.

In [ ]:
from pydantic import TypeAdapter
from stratifyx.types import ScreenAssetsRequest

cohort = await epoch.screen_assets(TypeAdapter(ScreenAssetsRequest).validate_python({
    "kind": "screen", "screen": {"all_of": [], "any_of": [], "index": ["S&P 500"], "index_membership_mode": "CurrentSnapshot"},
}))
print("Members:", cohort.asset_count)
print("Universe:", cohort.universe_ref)
print("Qualifications:", cohort.qualifications)
assert cohort.universe_ref is not None, "Read the native cohort gaps before using this universe."

## 4. Read the same saved search through a fresh MCP client

The MCP host loads the actual specialist catalog and reads the SDK-owned saved result. This step performs no new vendor search.

In [ ]:
import os
from mcp import Client
from epoch_sdk.agent_tools import AgentToolsClient
from epoch_sdk.mcp import create_server
from notebook_identity import identity_token

agent_tools = AgentToolsClient.connect(
    graph_url=os.environ["EPOCH_GRAPH_URL"], backend_url=os.environ["EPOCH_BACKEND_URL"], token=identity_token(),
)
try:
    async with Client(create_server(agent_tools)) as mcp:
        tools = (await mcp.list_tools()).tools
        names = {tool.name for tool in tools}
        assert "epoch_discover" not in names
        print("MCP tools:", sorted(names))
        page = await mcp.call_tool("epoch_read", {
            "kind": "dataset_search", "identifier": found.result.receipt.result_ref,
        })
        assert not page.is_error
        print("Saved rows read:", page.structured_content["returned"])
finally:
    await agent_tools.aclose()

In [ ]:
await epoch.aclose()